# 01 – Pembersihan dan Normalisasi Teks
Masukan: `songs_subset.parquet` (notebook 00). Keluaran: `songs_clean.parquet`, `lyrecall_text.py`, `clean_summary.json`.

`lyrecall_text.py` berisi fungsi normalisasi yang **dipakai bersama** oleh indeks (offline) dan kueri (online),
agar teks indeks dan teks kueri diproses dengan aturan yang sama.

In [1]:
# modul fungsi teks
from pathlib import Path

Path("/kaggle/working/lyrecall_text.py").write_text(r'''"""Fungsi teks bersama LyRecall: dipakai di indeks (offline) dan kueri (online)."""
import re
import unicodedata

# --- 1. Tabel penggantian karakter ---------------------------------------
_MOJIBAKE = {"â€™": "'", "â€˜": "'", "â€œ": '"', "â€\x9d": '"',
             "â€“": "-", "â€”": "-", "â€¦": "..."}
_MAP = {}
def _add(chars, rep):
    for ch in chars:
        _MAP[ord(ch)] = rep

_add("\u2019\u2018\u201b\u00b4\u2032\u0060\x91\x92", "'")        # apostrof/petik tunggal
_add("\u02bc\u02b9", "'")                                       # apostrof modifier
_add("\u2028\u2029\x85", "\n")                                  # pemisah baris Unicode
_add("\u201c\u201d\u201e\u2033\x93\x94", '"')                    # petik ganda
_add("\u201a", ",")
_add("\u2012\u2013\u2014\u2015\u2212\x96\x97", "-")              # strip
_add("\u2026", "...")
_add("".join(chr(c) for c in range(0x2000, 0x200B)) + "\u00a0\u202f\u205f\u3000", " ")  # spasi khusus
_add("\u200b\u200c\u200d\u2060\ufeff\u00ad", "")                 # karakter tak terlihat

# --- 2. Huruf Cyrillic yang menyerupai huruf Latin ------------------------
_HOMO = str.maketrans({
    "\u0430": "a", "\u0435": "e", "\u043e": "o", "\u0440": "p", "\u0441": "c",
    "\u0443": "y", "\u0445": "x", "\u0456": "i", "\u0458": "j", "\u0455": "s",
    "\u0410": "A", "\u0412": "B", "\u0415": "E", "\u041a": "K", "\u041c": "M",
    "\u041d": "H", "\u041e": "O", "\u0420": "P", "\u0421": "C", "\u0422": "T",
    "\u0425": "X", "\u0406": "I",
})
_RUN = re.compile(r"[^\W\d_]+")
_LAT = re.compile(r"[A-Za-z]")
_CYR = re.compile(r"[\u0400-\u04FF]")

def _fix_homoglyphs(text):
    if not _CYR.search(text):
        return text
    def f(m):
        w = m.group(0)
        return w.translate(_HOMO) if (_LAT.search(w) and _CYR.search(w)) else w
    return _RUN.sub(f, text)

# --- 3. Pembersihan lirik (untuk tampilan dan embedding) ------------------
_TAG = re.compile(r"\[[^\]\n]{0,100}\]")
_HAS_WORD = re.compile(r"\w")

def clean_lyrics(text):
    for k, v in _MOJIBAKE.items():
        text = text.replace(k, v)
    text = text.translate(_MAP)
    text = unicodedata.normalize("NFKC", text)
    text = _fix_homoglyphs(text)
    text = _TAG.sub("", text)
    lines = [re.sub(r"[ \t]+", " ", l).strip() for l in re.split(r"\r?\n", text)]
    return "\n".join(l for l in lines if l and _HAS_WORD.search(l))

# --- 4. Normalisasi leksikal (untuk BM25 dan TF-IDF; juga untuk kueri) ----
def normalize_lex(text):
    text = text.translate(_MAP)
    text = unicodedata.normalize("NFKD", text.lower())
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.replace("'", "")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

# --- 5. Jendela chunk -----------------------------------------------------
def make_chunks(n_lines, size, stride):
    if n_lines <= size:
        return [(0, n_lines)]
    starts = list(range(0, n_lines - size + 1, stride))
    if starts[-1] + size < n_lines:
        starts.append(n_lines - size)
    return [(s, s + size) for s in starts]
''', encoding="utf-8")

print("Tersimpan:", Path("/kaggle/working/lyrecall_text.py").stat().st_size, "byte")

Tersimpan: 3109 byte


In [2]:
import sys, os, re, glob, json, gc
import numpy as np, pandas as pd
from collections import Counter
import unicodedata
sys.path.insert(0, "/kaggle/working")
from lyrecall_text import clean_lyrics

SEED, MIN_WORDS, MIN_LINES, MAX_WORDS = 42, 20, 4, 1500
OUT_DIR = "/kaggle/working"

_HDR = re.compile(r"Lyrics\s*$")
def drop_header(text):
    """Buang baris pertama bila berupa header Genius ('... Lyrics')."""
    lines = text.split("\n")
    if lines and len(lines[0]) <= 120 and _HDR.search(lines[0]):
        lines = lines[1:]
    return "\n".join(lines)

path = glob.glob("/kaggle/input/datasets/nshamid/lyrecall-songs-subset/songs_subset.parquet", recursive=True)
print(path)
df = pd.read_parquet(path[0])
raw = df["lyrics"].copy()
df["lyrics"] = raw.apply(lambda t: drop_header(clean_lyrics(t)))
print("Lagu:", len(df))

['/kaggle/input/datasets/nshamid/lyrecall-songs-subset/songs_subset.parquet']
Lagu: 100000


In [3]:
# periksa hasil
for i in df.sample(2, random_state=SEED).index:
    print("SEBELUM:", repr(raw[i][:200]))
    print("SESUDAH:", repr(df.at[i, "lyrics"][:200]), "\n")

s = df["lyrics"].sample(20000, random_state=SEED)
c = Counter(ch for t in s for ch in t if ord(ch) > 127)
print("Non-ASCII tersisa:")
for ch, n in c.most_common(15):
    print(" ", repr(ch), hex(ord(ch)), unicodedata.name(ch, "?"), n)
print("\nMasih ada Cyrillic :", int(df["lyrics"].str.contains(r"[\u0400-\u04FF]").sum()))
print("Masih ada [penanda]:", int(df["lyrics"].str.contains(r"\[[^\]]+\]").sum()))

SEBELUM: "[Verse 1]\nI'm on the back of a new belief\nAnd on the back of a new belief\nMy friend rides all alone\nYeah, he's up and down like a new Jack Tone\nHe was alive in the new machine\nHe felt alive in the new"
SESUDAH: "I'm on the back of a new belief\nAnd on the back of a new belief\nMy friend rides all alone\nYeah, he's up and down like a new Jack Tone\nHe was alive in the new machine\nHe felt alive in the new machine\nM" 

SEBELUM: "[Intro]\nCarter in this bitch, now I feel like Nino\n(Free Numba 9 bitch, RIP LA, gang)\n\n[Pre-Hook: Doughboy]\n40 on my neck, now I'm the hood hero\nWe shooting bricks like we shooting a free throw\nEveryt"
SESUDAH: "Carter in this bitch, now I feel like Nino\n(Free Numba 9 bitch, RIP LA, gang)\n40 on my neck, now I'm the hood hero\nWe shooting bricks like we shooting a free throw\nEverything my way I swear I feel lik" 

Non-ASCII tersisa:
  'é' 0xe9 LATIN SMALL LETTER E WITH ACUTE 356
  'ñ' 0xf1 LATIN SMALL LETTER N WITH TILDE 105
  'à' 0xe0 L

In [4]:
# diagnosis panjang dan header “Lyrics”
df["n_lines"] = df["lyrics"].str.count("\n") + 1
df["n_words"] = df["lyrics"].str.count(r"\S+")
print(df[["n_lines", "n_words"]].describe(percentiles=[.01, .5, .99, .999]).round(1))
for w in (1000, 1500, 2000, 3000):
    print(f"> {w} kata:", int((df["n_words"] > w).sum()))
print("< MIN_LINES:", int((df["n_lines"] < MIN_LINES).sum()),
      "| < MIN_WORDS:", int((df["n_words"] < MIN_WORDS).sum()))

first = df["lyrics"].str.split("\n").str[0]
mask = first.str.contains(r"Lyrics\s*$")
print("\nBaris pertama berakhiran 'Lyrics':", int(mask.sum()))
for t in first[mask].head(8):
    print("  ", t[:100])

        n_lines   n_words
count  100000.0  100000.0
mean       41.9     286.2
std        25.2     193.6
min         1.0      17.0
1%          6.0      40.0
50%        37.0     242.0
99%       119.0     854.0
99.9%     202.0    1512.0
max       890.0    8045.0
> 1000 kata: 431
> 1500 kata: 103
> 2000 kata: 61
> 3000 kata: 32
< MIN_LINES: 66 | < MIN_WORDS: 10

Baris pertama berakhiran 'Lyrics': 0


In [5]:
# filter dan simpan
funnel = [("setelah pembersihan", len(df))]
d = df[df["n_lines"] >= MIN_LINES];  funnel.append((f">= {MIN_LINES} baris", len(d)))
d = d[d["n_words"] >= MIN_WORDS];    funnel.append((f">= {MIN_WORDS} kata", len(d)))
d = d[d["n_words"] <= MAX_WORDS];    funnel.append((f"<= {MAX_WORDS} kata", len(d)))
for n_, k in funnel:
    print(f"{n_:26s}: {k:>9,}")

d[["song_id", "title", "artist", "lyrics", "n_lines", "n_words"]].to_parquet(
    f"{OUT_DIR}/songs_clean.parquet", index=False)
with open(f"{OUT_DIR}/clean_summary.json", "w") as f:
    json.dump({"funnel": funnel, "max_words": MAX_WORDS}, f, indent=2)
print("\nTersimpan:", len(d), "lagu (song_id dipertahankan, tidak diberi nomor ulang)")

setelah pembersihan       :   100,000
>= 4 baris                :    99,934
>= 20 kata                :    99,924
<= 1500 kata              :    99,821

Tersimpan: 99821 lagu (song_id dipertahankan, tidak diberi nomor ulang)
